# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrishagk/My_flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Rank content items higher when multiple observed signals indicate potential review value. The baseline gives 3 points for a short-window observed decline in clicks, 2 points for relatively low CTR, 2 points for relatively weak average search position, and 1 point for relatively low organic-session share when that signal is available. The score is a decision-support ranking for human review, not an automatic refresh decision.

Reason codes: TREND_DOWN, LOW_CTR, WEAK_POSITION, and LOW_ORGANIC_SHARE.

The January data contains five daily observations, so TREND_DOWN represents an observed change between the earliest and latest available dates rather than a proven long-term decline

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
from getpass import getpass
import duckdb

HF_TOKEN = getpass("Enter your Hugging Face token: ")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

In [4]:

REL = "hf://datasets/FlyRank/internship-warehouse"

PERF_JAN = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2025-01/*.parquet'"
    f")"
)

print(con.sql(f"SELECT * FROM {PERF_JAN} LIMIT 5").df())

  report_date           client_hash_id           content_hash_id  \
0  2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb   
1  2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2   
2  2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058   
3  2025-01-27  client_9958f0a7ae1df715  content_c899aef92518c714   
4  2025-01-27  client_9958f0a7ae1df715  content_c7c1d2e68d9d0964   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True            True                True               False   
1            True            True                True               False   
2            True            True                True               False   
3            True            True                True               False   
4            True            True                True               False   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               30           0               115  ...     

In [5]:

jan_sample = con.sql(f'SELECT * FROM {PERF_JAN}').df()

print(jan_sample.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [11]:
# Work from the original daily observations.
trend_df = jan_sample.copy()

trend_df["report_date"] = pd.to_datetime(trend_df["report_date"])

# Earliest and latest dates available.
first_date = trend_df["report_date"].min()
latest_date = trend_df["report_date"].max()

print("First date:", first_date.date())
print("Latest date:", latest_date.date())

# Get clicks on the first available date.
first_clicks = (
    trend_df[trend_df["report_date"] == first_date]
    [["client_hash_id", "content_hash_id", "gsc_clicks"]]
    .rename(columns={"gsc_clicks": "first_clicks"})
)

# Get clicks on the latest available date.
latest_clicks = (
    trend_df[trend_df["report_date"] == latest_date]
    [["client_hash_id", "content_hash_id", "gsc_clicks"]]
    .rename(columns={"gsc_clicks": "latest_clicks"})
)

# Join the two dates for each content item.
trend = first_clicks.merge(
    latest_clicks,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Calculate percentage change.
trend["click_change_pct"] = np.where(
    trend["first_clicks"] > 0,
    (trend["latest_clicks"] - trend["first_clicks"])
    / trend["first_clicks"],
    np.nan
)

# Short-window observed decline.
trend["trend_down"] = trend["click_change_pct"] < 0

display(trend.head())

First date: 2025-01-27
Latest date: 2025-01-31


,client_hash_id,content_hash_id,first_clicks,latest_clicks,click_change_pct,trend_down
0,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,0,0,NaN,False
1,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,0,0,NaN,False
2,client_9958f0a7ae1df715,content_b4462a1b90640058,0,0,NaN,False
3,client_9958f0a7ae1df715,content_c899aef92518c714,0,0,NaN,False
4,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,0,0,NaN,False


In [12]:
baseline = jan_sample.copy()

baseline["report_date"] = pd.to_datetime(baseline["report_date"])

# Only score the latest available date.
baseline = baseline[
    baseline["report_date"] == latest_date
].copy()

# Add the short-window trend.
baseline = baseline.merge(
    trend[
        [
            "client_hash_id",
            "content_hash_id",
            "first_clicks",
            "latest_clicks",
            "click_change_pct",
            "trend_down",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print("Rows scored:", len(baseline))
display(baseline.head())

Rows scored: 221


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,first_clicks,latest_clicks,click_change_pct,trend_down
0,2025-01-31,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,24,0,149,...,0,0,0,0,0,2025-01,0.0,0.0,NaN,False
1,2025-01-31,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,6,0,275,...,0,0,0,0,0,2025-01,0.0,0.0,NaN,False
2,2025-01-31,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,2,0,21,...,0,0,0,0,0,2025-01,0.0,0.0,NaN,False
3,2025-01-31,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,7,0,187,...,0,0,0,0,0,2025-01,0.0,0.0,NaN,False
4,2025-01-31,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,10,0,291,...,0,0,0,0,0,2025-01,0.0,0.0,NaN,False


In [13]:
baseline["ctr"] = np.where(
    baseline["gsc_impressions"] > 0,
    baseline["gsc_clicks"] / baseline["gsc_impressions"],
    np.nan
)

baseline["total_sessions"] = (
    baseline["sessions_organic"].fillna(0)
    + baseline["sessions_direct"].fillna(0)
    + baseline["sessions_referral"].fillna(0)
    + baseline["sessions_social"].fillna(0)
    + baseline["sessions_paid"].fillna(0)
    + baseline["sessions_ai"].fillna(0)
)

baseline["organic_share"] = np.where(
    baseline["total_sessions"] > 0,
    baseline["sessions_organic"] / baseline["total_sessions"],
    np.nan
)

In [14]:
ctr_threshold = baseline["ctr"].quantile(0.25)

position_threshold = baseline["gsc_avg_position"].quantile(0.75)

organic_share_threshold = baseline["organic_share"].quantile(0.25)

print("CTR threshold:", ctr_threshold)
print("Position threshold:", position_threshold)
print("Organic share threshold:", organic_share_threshold)

CTR threshold: 0.0
Position threshold: 49.666666666666664
Organic share threshold: nan


In [16]:
baseline["score"] = 0
baseline["reason_codes"] = ""

# ---------------------------------------------------------
# Boolean conditions
# ---------------------------------------------------------

# Only treat a measurable negative change as a decline.
trend_down = (
    baseline["trend_down"]
    .fillna(False)
    .astype(bool)
)

# Missing CTR should not automatically count as LOW_CTR.
low_ctr = (
    (baseline["ctr"] <= ctr_threshold)
    .fillna(False)
    .astype(bool)
)

# Missing position should not automatically count as WEAK_POSITION.
weak_position = (
    (baseline["gsc_avg_position"] >= position_threshold)
    .fillna(False)
    .astype(bool)
)

# Missing organic share should not automatically count as LOW_ORGANIC_SHARE.
low_organic_share = (
    (baseline["organic_share"] <= organic_share_threshold)
    .fillna(False)
    .astype(bool)
)

# ---------------------------------------------------------
# Add points
# ---------------------------------------------------------

baseline.loc[trend_down, "score"] += 3
baseline.loc[low_ctr, "score"] += 2
baseline.loc[weak_position, "score"] += 2
baseline.loc[low_organic_share, "score"] += 1

# ---------------------------------------------------------
# Add reason codes
# ---------------------------------------------------------

baseline.loc[trend_down, "reason_codes"] += "TREND_DOWN;"
baseline.loc[low_ctr, "reason_codes"] += "LOW_CTR;"
baseline.loc[weak_position, "reason_codes"] += "WEAK_POSITION;"
baseline.loc[low_organic_share, "reason_codes"] += "LOW_ORGANIC_SHARE;"

print("Scoring completed.")
print("Rows scored:", len(baseline))
print("Maximum score:", baseline["score"].max())
print("Minimum score:", baseline["score"].min())

Scoring completed.
Rows scored: 221
Maximum score: 5
Minimum score: 0


In [17]:
queue = baseline.sort_values(
    ["score", "click_change_pct"],
    ascending=[False, True]
).copy()

queue["rank"] = range(1, len(queue) + 1)

display(
    queue[
        [
            "rank",
            "content_hash_id",
            "score",
            "reason_codes",
            "click_change_pct",
            "ctr",
            "gsc_avg_position",
            "organic_share",
        ]
    ].head(20)
)

,rank,content_hash_id,score,reason_codes,click_change_pct,ctr,gsc_avg_position,organic_share
19,1,content_d237e43c93144a57,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,21.555556,NaN
54,2,content_c0cc958ff5d50e85,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,33.222222,NaN
75,3,content_de7b08874af74c00,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,6.352941,NaN
115,4,content_89fa653709e53ad8,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,4.500000,NaN
133,5,content_1b1f69effd0e4531,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,8.083333,NaN
138,6,content_1c7ae2a4d8295008,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,4.037037,NaN
140,7,content_dc40416a3cf75e56,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,2.133333,NaN
5,8,content_c782fa8abd4fce5e,4,LOW_CTR;WEAK_POSITION;,NaN,0.0,51.285714,NaN
12,9,content_89c6c2e17e412e20,4,LOW_CTR;WEAK_POSITION;,NaN,0.0,61.666667,NaN
16,10,content_6f5cc2a41d94e372,4,LOW_CTR;WEAK_POSITION;,NaN,0.0,52.607143,NaN


In [19]:
from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

queue_output.to_csv(output_path, index=False)

print(f"Wrote {len(queue_output)} rows to {output_path}")
display(queue_output.head(20))

Wrote 221 rows to work\outputs\baseline_action_score.csv


,rank,content_hash_id,score,reason_codes,click_change_pct,ctr,gsc_avg_position,organic_share
19,1,content_d237e43c93144a57,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,21.555556,NaN
54,2,content_c0cc958ff5d50e85,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,33.222222,NaN
75,3,content_de7b08874af74c00,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,6.352941,NaN
115,4,content_89fa653709e53ad8,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,4.500000,NaN
133,5,content_1b1f69effd0e4531,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,8.083333,NaN
138,6,content_1c7ae2a4d8295008,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,4.037037,NaN
140,7,content_dc40416a3cf75e56,5,TREND_DOWN;LOW_CTR;,-1.0,0.0,2.133333,NaN
5,8,content_c782fa8abd4fce5e,4,LOW_CTR;WEAK_POSITION;,NaN,0.0,51.285714,NaN
12,9,content_89c6c2e17e412e20,4,LOW_CTR;WEAK_POSITION;,NaN,0.0,61.666667,NaN
16,10,content_6f5cc2a41d94e372,4,LOW_CTR;WEAK_POSITION;,NaN,0.0,52.607143,NaN


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
top20 = queue_output.head(20).copy()

top20["action"] = "REVIEW"

top20["confidence_note"] = top20["reason_codes"].apply(
    lambda x:
        "Moderate: multiple observed performance signals support review."
        if ";" in x.strip(";")
        else
        "Lower: only one observed signal supports review."
)

top20["what_would_make_it_wrong"] = (
    "The short 5-day window may not represent a sustained pattern; "
    "zero clicks may reflect temporary or measurement conditions."
)

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_codes",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]

display(top20_review)

,rank,content_hash_id,action,reason_codes,confidence_note,what_would_make_it_wrong
19,1,content_d237e43c93144a57,REVIEW,TREND_DOWN;LOW_CTR;,Moderate: multiple observed performance signal...,The short 5-day window may not represent a sus...
54,2,content_c0cc958ff5d50e85,REVIEW,TREND_DOWN;LOW_CTR;,Moderate: multiple observed performance signal...,The short 5-day window may not represent a sus...
75,3,content_de7b08874af74c00,REVIEW,TREND_DOWN;LOW_CTR;,Moderate: multiple observed performance signal...,The short 5-day window may not represent a sus...
115,4,content_89fa653709e53ad8,REVIEW,TREND_DOWN;LOW_CTR;,Moderate: multiple observed performance signal...,The short 5-day window may not represent a sus...
133,5,content_1b1f69effd0e4531,REVIEW,TREND_DOWN;LOW_CTR;,Moderate: multiple observed performance signal...,The short 5-day window may not represent a sus...
138,6,content_1c7ae2a4d8295008,REVIEW,TREND_DOWN;LOW_CTR;,Moderate: multiple observed performance signal...,The short 5-day window may not represent a sus...
140,7,content_dc40416a3cf75e56,REVIEW,TREND_DOWN;LOW_CTR;,Moderate: multiple observed performance signal...,The short 5-day window may not represent a sus...
5,8,content_c782fa8abd4fce5e,REVIEW,LOW_CTR;WEAK_POSITION;,Moderate: multiple observed performance signal...,The short 5-day window may not represent a sus...
12,9,content_89c6c2e17e412e20,REVIEW,LOW_CTR;WEAK_POSITION;,Moderate: multiple observed performance signal...,The short 5-day window may not represent a sus...
16,10,content_6f5cc2a41d94e372,REVIEW,LOW_CTR;WEAK_POSITION;,Moderate: multiple observed performance signal...,The short 5-day window may not represent a sus...


The top 20 are treated as review candidates, not automatic refresh decisions. The strongest candidates have multiple observed signals, while the short five-day history limits confidence. A page could be a weak pick if the apparent decline is temporary, if zero clicks reflects measurement or coverage conditions, or if additional business/content context would change the decision.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some top picks are potentially weak because the baseline uses only five consecutive days of performance. A page with zero clicks on the latest day can receive TREND_DOWN even though the decline may be temporary. Pages ranked from LOW_CTR + WEAK_POSITION may also need additional content or business context before a refresh decision is made.

The baseline does not use product flags or future outcome-window fields. The trend feature is constructed from the earliest and latest observations available in the current historical slice. Therefore, the score is intended as a directional, decision-support baseline rather than a causal prediction.

In [21]:
# ---------------------------------------------------------
# Leakage / feature check
# ---------------------------------------------------------

used_features = [
    "trend_down",
    "ctr",
    "gsc_avg_position",
    "organic_share",
]

print("Features used by baseline:")
print(used_features)

# Check that identifiers were not used as predictive features.
identifier_features = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
]

print("\nIdentifiers excluded from scoring:")
print(identifier_features)

# Check that no obvious future-label field is present.
future_like_fields = [
    col for col in baseline.columns
    if any(term in col.lower() for term in [
        "future",
        "outcome",
        "label",
        "target"
    ])
]

print("\nPotential future/label-like fields present in baseline:")
print(future_like_fields)

# Confirm the trend uses only the available historical window.
print("\nTrend window:")
print("First available date:", first_date.date())
print("Latest available date:", latest_date.date())

assert "score" in baseline.columns
assert "reason_codes" in baseline.columns

print("\nBaseline leakage checks completed.")

Features used by baseline:
['trend_down', 'ctr', 'gsc_avg_position', 'organic_share']

Identifiers excluded from scoring:
['client_hash_id', 'content_hash_id', 'report_date']

Potential future/label-like fields present in baseline:
[]

Trend window:
First available date: 2025-01-27
Latest available date: 2025-01-31

Baseline leakage checks completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.